In [2]:
import pandas as pd
import plotly.express as px
df = pd.read_csv("ai_job_impact.csv", index_col=0)
df.head()



,Age,Gender,Education_Level,Industry,Job_Role,Years_Experience,AI_Adoption_Level,Automation_Risk,Upskilling_Required,Salary_Before_AI,Salary_After_AI,Job_Status,Work_Hours_Per_Week,Remote_Work,Job_Satisfaction,Productivity_Change_%
Employee_ID,,,,,,,,,,,,,,,,
E0001,50,Female,Bachelor,Marketing,Content Creator,26,High,High,Yes,106820,95455,Replaced,45,No,5,-10.64
E0002,45,Male,High School,Manufacturing,Quality Inspector,19,Low,Low,Yes,74131,72013,Unchanged,36,Yes,6,19.05
E0003,51,Female,Master,IT,DevOps Engineer,28,Medium,Medium,Yes,35311,42290,Modified,46,Yes,3,17.05
E0004,48,Male,PhD,Education,Teacher,24,Medium,Medium,Yes,114478,107820,Modified,50,No,9,-2.47
E0005,24,Male,Bachelor,Healthcare,Doctor,0,High,Medium,No,33890,40945,Modified,52,Yes,6,7.03


In [7]:
# 1. Calculate the percentage of 'High' for Adoption
adopt_high = df[df['AI_Adoption_Level'] == 'High']['Industry'].value_counts()
total_workers = df['Industry'].value_counts()
adopt_pct = (adopt_high / total_workers * 100).fillna(0)

# 2. Calculate the percentage of 'High' for Risk
risk_high = df[df['Automation_Risk'] == 'High']['Industry'].value_counts()
risk_pct = (risk_high / total_workers * 100).fillna(0)

# 3. Combine into a single DataFrame
compare_df = pd.DataFrame({
    'High Adoption': adopt_pct, 
    'High Risk': risk_pct
}).reset_index()
compare_df.columns = ['Industry', 'High Adoption', 'High Risk']

# 4. Sort by High Adoption so the chart looks organized
compare_df = compare_df.sort_values(by='High Adoption', ascending=True)

# 5. 'Melt' the data for Plotly Express (turns wide columns into rows)
melted_df = compare_df.melt(
    id_vars='Industry', 
    value_vars=['High Adoption', 'High Risk'], 
    var_name='Metric', 
    value_name='Percentage'
)

# 6. Create the Grouped Horizontal Bar Chart
fig = px.bar(
    melted_df, 
    x='Percentage', 
    y='Industry', 
    color='Metric', 
    barmode='group', # Puts the bars side-by-side
    orientation='h',
    text_auto='.1f',
    title="Industry Leaders: High AI Adoption vs. High Automation Risk",
    color_discrete_map={
        'High Adoption': '#3498DB', # Blue for Adoption
        'High Risk': '#E74C3C'      # Red for Risk
    },
    template="plotly_white",
    height=600
)

# 7. Clean up the layout
fig.update_layout(
    xaxis_title="Percentage of Workforce (%)",
    yaxis_title="Industry Sector",
    legend_title="Metric"
)

fig.show()

In [3]:
orders = {
    "AI_Adoption_Level": ["Low", "Medium", "High"],
    "Automation_Risk": ["Low", "Medium", "High"],
    "Job_Status": ["Unchanged", "Modified", "Replaced"]
}

# 2. Define colors for the Outcome (Job Status)
# We use Green for Unchanged, Yellow for Modified, Red for Replaced
status_colors = {
    "Unchanged": "#2ECC71", 
    "Modified": "#F1C40F", 
    "Replaced": "#E74C3C"
}

# 3. Create the Faceted Histogram
fig = px.histogram(
    df, 
    x="Automation_Risk", 
    color="Job_Status",
    facet_col="AI_Adoption_Level", # This creates the 3 panels
    category_orders=orders,
    color_discrete_map=status_colors,
    barnorm='percent',              # Shows the ratio/percentage
    text_auto='.0f',               # Adds the % numbers on the bars
    title="The Disruption Matrix: How Adoption & Risk Predict Job Status",
    template="plotly_white",
    height=500
)

# 4. Improve the look
fig.update_layout(
    xaxis_title="Automation Risk",
    yaxis_title="Percentage of Jobs (%)",
    legend_title="Outcome (Job Status)"
)

# This cleans up the titles above each panel (e.g., "AI_Adoption_Level=High" -> "High Adoption")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1] + " Adoption"))

fig.show()

In [4]:


# 1. Calculate the Replacement Rate per Industry
# We create a df of Job_Status percentages for each Industry
replacement_stats = pd.crosstab(df['Industry'], df['Job_Status'], normalize='index') * 100

# 2. Convert to a clean DataFrame and sort by 'Replaced'
replacement_df = replacement_stats['Replaced'].reset_index()
replacement_df.columns = ['Industry', 'Replacement_Rate']
replacement_df = replacement_df.sort_values(by='Replacement_Rate', ascending=True) # Ascending for a nice horizontal bar flow

# 3. Create the Visualization
fig = px.bar(
    replacement_df, 
    x='Replacement_Rate', 
    y='Industry', 
    orientation='h',
    text_auto='.1f',
    title="The Replacement Leaderboard: % of Workforce Replaced by AI",
    labels={'Replacement_Rate': 'Replacement Rate (%)'},
    template="plotly_white",
    color='Replacement_Rate',
    # Using a Red sequential scale to signal 'Danger/Displacement'
    color_continuous_scale=['#FFC0B5', '#E74C3C'] 
)

# 4. Final Polish
fig.update_layout(
    xaxis_title="Percentage of Industry Displaced (%)",
    yaxis_title="Sector",
    coloraxis_showscale=False # Hides the side color bar for a cleaner look
)

fig.show()


In [4]:
# 1. Filter for 'Modified' jobs only
mod_df = df[df['Job_Status'] == 'Modified']

# 2. Calculate Total Workers per Industry (to get the ratio)
total_counts = df['Industry'].value_counts().reset_index()
total_counts.columns = ['Industry', 'Total_Workers']

# 3. Calculate Average Productivity AND Count of Modified Workers
mod_stats = mod_df.groupby('Industry').agg(
    Modified_Count=('Job_Status', 'count'),  # <--- Safely counts the rows using a known column
    Avg_Productivity=('Productivity_Change_%', 'mean')
).reset_index()
# 4. Merge and Calculate the Modification Ratio (%)
mod_merged = pd.merge(mod_stats, total_counts, on='Industry')
mod_merged['Modification_Rate'] = (mod_merged['Modified_Count'] / mod_merged['Total_Workers']) * 100

# 5. Create the Scatter Plot (Bubble Chart)
fig = px.scatter(
    mod_merged, 
    x='Modification_Rate', 
    y='Avg_Productivity', 
    size='Total_Workers', # Bubble size represents the size of the industry
    color='Industry',
    text='Industry',
    title="The Transformation Quadrant: Modification Rate vs. Productivity Gains",
    labels={
        'Modification_Rate': 'Workforce Modified (%)',
        'Avg_Productivity': 'Average Productivity Change (%)'
    },
    template="plotly_white"
)

# 6. Polish the chart (push text labels slightly above bubbles)
fig.update_traces(textposition='top center')

# Optional: Add dashed lines to show the average across ALL industries
fig.add_hline(y=mod_merged['Avg_Productivity'].mean(), line_dash="dot", line_color="gray", annotation_text="Avg Productivity Gain")
fig.add_vline(x=mod_merged['Modification_Rate'].mean(), line_dash="dot", line_color="gray", annotation_text="Avg Modification Rate")

fig.show()

In [11]:
# 1. Filter for 'Modified' jobs only
mod_df = df[df['Job_Status'] == 'Unchanged']  # <--- This to show what sectors are resisting change and how that correlates with productivity (could be interesting to show both Modified and Unchanged in the final portfolio)

# 2. Calculate Total Workers per Industry (to get the ratio)
total_counts = df['Industry'].value_counts().reset_index()
total_counts.columns = ['Industry', 'Total_Workers']

# 3. Calculate Average Productivity AND Count of Modified Workers
mod_stats = mod_df.groupby('Industry').agg(
    Modified_Count=('Job_Status', 'count'),  # <--- Safely counts the rows using a known column
    Avg_Productivity=('Productivity_Change_%', 'mean')
).reset_index()
# 4. Merge and Calculate the Modification Ratio (%)
mod_merged = pd.merge(mod_stats, total_counts, on='Industry')
mod_merged['Modification_Rate'] = (mod_merged['Modified_Count'] / mod_merged['Total_Workers']) * 100

# 5. Create the Scatter Plot (Bubble Chart)
fig = px.scatter(
    mod_merged, 
    x='Modification_Rate', 
    y='Avg_Productivity', 
    size='Total_Workers', # Bubble size represents the size of the industry
    color='Industry',
    text='Industry',
    title="The Transformation Quadrant: Unchanged Workforce vs. Productivity Gains",
    labels={
        'Modification_Rate': 'Workforce Unchanged (%)',
        'Avg_Productivity': 'Average Productivity Change (%)'
    },
    template="plotly_white"
)

# 6. Polish the chart (push text labels slightly above bubbles)
fig.update_traces(textposition='top center')

# Optional: Add dashed lines to show the average across ALL industries
fig.add_hline(y=mod_merged['Avg_Productivity'].mean(), line_dash="dot", line_color="gray", annotation_text="Avg Productivity Gain")
fig.add_vline(x=mod_merged['Modification_Rate'].mean(), line_dash="dot", line_color="gray", annotation_text="Avg Modification Rate")

fig.show()

In [8]:


# 1. Define your mapping dictionary
mapping = {'Low': 1, 'Medium': 3, 'High': 5}

# 2. Map the text to numbers
df['Adoption_Num'] = df['AI_Adoption_Level'].map(mapping)
df['Risk_Num'] = df['Automation_Risk'].map(mapping)

# 3. Combine them using Multiplication (The Disruption Score)
df['AI_Disruption_Score'] = df['Adoption_Num'] * df['Risk_Num']

# 4. Prove the insight: Does a higher score mean more productivity, or just more chaos?
fig = px.scatter(
    df,
    x='AI_Disruption_Score',
    y='Productivity_Change_%',
    color='Job_Status',
    opacity=0.7,
    trendline="ols", # Adds a line of best fit to prove the mathematical trend!
    title="The Gravity of Disruption: Impact Score vs. Productivity",
    template="plotly_white",
    color_discrete_map={"Unchanged": "#2ECC71", "Modified": "#F1C40F", "Replaced": "#E74C3C"}
)

fig.update_layout(xaxis_title="AI Disruption Score (1-25)", yaxis_title="Productivity Change (%)")
fig.show()